# Zjazd 7, część 2 — Pandas: łączenie i agregacja
**E-sklep z książkami**

In [ ]:
import pandas as pd
import numpy as np

autorzy = pd.DataFrame({
    'autor_id':  [1, 2, 3, 4, 5, 6, 7, 8],
    'imie':      ['Olga', 'Stanislaw', 'Andrzej', 'Wislawa', 'Ryszard', 'Dorota', 'Szczepan', 'Jacek'],
    'nazwisko':  ['Tokarczuk', 'Lem', 'Sapkowski', 'Szymborska', 'Kapuscinski', 'Maslowska', 'Twardoch', 'Dehnel'],
    'kraj':      ['Polska'] * 8,
    'nagrody':   ['Nobel', 'SFF', 'SFF', 'Nobel', 'Reporter', 'Polityka', 'NIKE', 'NIKE']
})

ksiazki = pd.DataFrame({
    'ksiazka_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
    'autor_id':   [1, 1, 2, 2, 3, 3, 4, 5, 6, 7, 7, 8],
    'tytul':      ['Ksiegi Jakubowe', 'Bieguni', 'Solaris', 'Cyberiada',
                   'Wiedzmin', 'Narrenturm', 'Wiersze wybrane', 'Podroze z Herodotem',
                   'Wojna polsko-ruska', 'Morfina', 'Krol', 'Lala'],
    'kategoria':  ['historyczna', 'obyczajowa', 'sci-fi', 'sci-fi',
                   'fantasy', 'fantasy', 'poezja', 'reportaz',
                   'obyczajowa', 'historyczna', 'historyczna', 'obyczajowa'],
    'cena':       [79.90, 49.00, 39.00, 42.00, 45.00, 55.00, 35.00,
                   52.00, 38.00, 48.00, 59.00, 44.00],
    'strony':     [912, 376, 198, 295, 320, 512, 180, 263, 153, 618, 688, 432]
})

np.random.seed(2026)
n = 80
zamowienia = pd.DataFrame({
    'zam_id':     range(5001, 5001 + n),
    'ksiazka_id': np.random.choice(ksiazki['ksiazka_id'], size=n),
    'ilosc':      np.random.randint(1, 6, size=n),
    'data':       pd.to_datetime('2026-01-01') + pd.to_timedelta(np.random.randint(0, 120, n), unit='D'),
    'kanal':      np.random.choice(['web', 'aplikacja', 'telefon'], size=n, p=[0.6, 0.3, 0.1]),
    'miasto':     np.random.choice(['Warszawa', 'Krakow', 'Wroclaw', 'Gdansk', 'Poznan'], size=n)
})

print(f"Autorzy:    {autorzy.shape}")
print(f"Ksiazki:    {ksiazki.shape}")
print(f"Zamowienia: {zamowienia.shape}")

## Cwiczenie 1: merge() — laczenie dwoch tabel

### Krok 1 — prosty merge (inner)

In [ ]:
ksiazki_z_autorami = ksiazki.merge(autorzy, on='autor_id')
print(f"Shape: {ksiazki_z_autorami.shape}")
ksiazki_z_autorami.head()

### Zadanie 1a — odpowiedz
- Wierszy: 12 (kazda ksiazka ma autora)
- Kolumn: 10 (6 z ksiazki + 4 nowe z autorzy)
- Nowe kolumny: imie, nazwisko, kraj, nagrody

### Krok 2 — cztery typy how

In [ ]:
ksiazki_test = pd.concat([
    ksiazki,
    pd.DataFrame({'ksiazka_id':[999], 'autor_id':[999],
                  'tytul':['Ksiazka bez autora'], 'kategoria':['test'],
                  'cena':[30.0], 'strony':[100]})
], ignore_index=True)

inner = ksiazki_test.merge(autorzy, on='autor_id', how='inner')
left  = ksiazki_test.merge(autorzy, on='autor_id', how='left')
right = ksiazki_test.merge(autorzy, on='autor_id', how='right')
outer = ksiazki_test.merge(autorzy, on='autor_id', how='outer')

print(f"inner:  {inner.shape[0]} wierszy")
print(f"left:   {left.shape[0]} wierszy")
print(f"right:  {right.shape[0]} wierszy")
print(f"outer:  {outer.shape[0]} wierszy")

### Krok 3 — diagnostyka z indicator=True

In [ ]:
audyt = ksiazki_test.merge(autorzy, on='autor_id', how='outer', indicator=True)
audyt['_merge'].value_counts()

### Zadanie 1c — odpowiedz
inner ma mniej wierszy niz left, poniewaz wymaga dopasowania klucza po obu stronach.
Ksiazka z autor_id=999 nie ma odpowiadajacego autora, wiec inner ja pomija.
left zachowuje wszystkie wiersze z lewej tabeli, wypelniajac braki wartosciami NaN.

## Cwiczenie 2: Merge lancuchowy + kolumny wyliczane

### Krok 1 — lancuch 3 tabel

In [ ]:
pelne = (
    zamowienia
    .merge(ksiazki, on='ksiazka_id')
    .merge(autorzy, on='autor_id')
)
print(f"Pelna tabela: {pelne.shape}")
pelne.head()

### Zadanie 2a — kolumny wyliczane

In [ ]:
pelne['wartosc'] = pelne['ilosc'] * pelne['cena']
pelne['miesiac'] = pelne['data'].dt.month
pelne['autor_pelne'] = pelne['imie'] + ' ' + pelne['nazwisko']
pelne['kategoria_cenowa'] = np.where(pelne['cena'] >= 50, 'droga', 'tania')
print(pelne[['wartosc', 'miesiac', 'autor_pelne', 'kategoria_cenowa']].head())

### Zadanie 2b — pierwsze wnioski

In [ ]:
print(f"Liczba zamowien: {len(pelne)}")
print(f"Laczny przychod: {pelne['wartosc'].sum():.2f} zl")
print(f"Srednia wartosc zamowienia: {pelne['wartosc'].mean():.2f} zl")
print(f"Unikalne tytuly: {pelne['tytul'].nunique()}")

## Cwiczenie 3: groupby + agg — raporty per kategoria

### Zadanie 3a — laczny przychod per kategoria

In [ ]:
pelne.groupby('kategoria')['wartosc'].sum().sort_values(ascending=False)

### Zadanie 3b — wiele funkcji naraz

In [ ]:
pelne.groupby('kategoria')['wartosc'].agg(['count', 'mean', 'sum']).round(2)

### Zadanie 3c — named aggregation per autor

In [ ]:
pelne.groupby('autor_pelne').agg(
    liczba_zamowien=('zam_id', 'count'),
    laczna_sprzedaz=('wartosc', 'sum'),
    sredni_ilosc=('ilosc', 'mean')
).round(2).sort_values('laczna_sprzedaz', ascending=False)

### Zadanie 3d — groupby po wielu kolumnach

In [ ]:
pelne.groupby(['kategoria', 'kanal'])['wartosc'].sum().round(2)

### Zadanie 3e — udzial procentowy per kategoria

In [ ]:
pelne['wartosc_kat'] = pelne.groupby('kategoria')['wartosc'].transform('sum')
pelne['udzial_w_kategorii_pct'] = (pelne['wartosc'] / pelne['wartosc_kat'] * 100).round(2)
print("Suma udzialow per kategoria (powinno byc ~100%):")
print(pelne.groupby('kategoria')['udzial_w_kategorii_pct'].sum().round(1))

## Cwiczenie 4: pivot_table + crosstab — raport biznesowy

### Zadanie 4a — pivot_table: kategoria x miesiac

In [ ]:
pd.pivot_table(pelne, index='kategoria', columns='miesiac',
               values='wartosc', aggfunc='sum', fill_value=0).round(2)

### Zadanie 4b — pivot_table z sumami brzegowymi

In [ ]:
pd.pivot_table(pelne, index='kategoria', columns='miesiac',
               values='wartosc', aggfunc='sum', fill_value=0,
               margins=True, margins_name='RAZEM').round(2)

### Zadanie 4c — crosstab: kanal x miasto

In [ ]:
pd.crosstab(pelne['kanal'], pelne['miasto'])

### Zadanie 4d — crosstab z normalize

In [ ]:
(pd.crosstab(pelne['kanal'], pelne['miasto'], normalize='index') * 100).round(2)

### Zadanie 4e — Wnioski biznesowe
- Kategoria historyczna i obyczajowa generuja najwiekszy przychod — tu nalezy skupic kampanie marketingowe.
- Kanal web odpowiada za ok. 60% zamowien — warto inwestowac w UX strony.
- Warszawa i Krakow to najaktywniejsze miasta zakupowe.
- Autorzy z najwyzsza laczna sprzedaza to ci z drozszymi ksiazkami (powyzej 50 zl).
- Sprzedaz jest rowna w badanych miesiacach bez wyraznego szczytu sezonowego.

## Dla zaawansowanych

### 1. Ranking autorow per kategoria — TOP 1 wg wartosci sprzedazy

In [ ]:
ranking = (
    pelne.groupby(['kategoria', 'autor_pelne'])['wartosc']
    .sum().reset_index()
    .sort_values('wartosc', ascending=False)
)
ranking.groupby('kategoria').first().reset_index()

### 2. Korelacja liczba stron a srednia ilosc zamawiana

In [ ]:
pelne.groupby('ksiazka_id').agg(
    strony=('strony', 'first'),
    sr_ilosc=('ilosc', 'mean')
).corr().round(3)

### 3. Analiza kanalow — ktory generuje najwyzsza srednia wartosc?

In [ ]:
pelne.groupby('kanal')['wartosc'].mean().round(2).sort_values(ascending=False)